# **Step 1: Protein Preparation (8RRJ, remove ligand + NADPH, auto-extract reference ligand)**

This is a minimal-change adaptation of your original protein-preparation workflow for **8RRJ**:

- fetches **8RRJ** by default
- **automatically extracts the crystallized small-molecule ligand** from the raw structure for later box definition
- removes **all heterogens and waters** during APO preparation, so the receptor is clean
- keeps the original backbone of the workflow: **PDBFixer -> BioPython -> PropKa -> pdb2pqr -> OpenMM EM**


In [ ]:

# =============================================================================
#  Protein Preparation Pipeline for Molecular Docking (8RRJ version)
# =============================================================================
#  Adapted from the original notebook with minimal logic changes:
#    - default structure = 8RRJ
#    - auto-extract reference ligand from raw structure before APO conversion
#    - remove ligand + NADPH during protein preparation
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

structure_source = "Fetch from RCSB PDB"  # @param ["Upload PDB file", "Fetch from RCSB PDB"]
pdb_id = "8RRJ"  # @param {type:"string"}
target_pH = 7.4  # @param {type:"number"}
restraint_k = 500  # @param {type:"number"}
convergence_tol = 10  # @param {type:"number"}
max_iterations = 1  # @param {type:"integer"}
reference_ligand_resname = "auto"  # @param {type:"string"}
cofactor_names_csv = "NAP,NAD,NADP,NDP,NADPH"  # @param {type:"string"}


# ---------------------------------------------------------------------------
#  STEP 0 : Install Dependencies
# ---------------------------------------------------------------------------

import subprocess, sys

print("=" * 72)
print("  STEP 0 | Installing Dependencies")
print("=" * 72)

_packages = {
    "openmm":    "openmm",
    "pdbfixer":  "pdbfixer",
    "biopython": "Bio",
    "propka":    "propka",
    "pdb2pqr":   "pdb2pqr",
    "requests":  "requests",
    "numpy":     "numpy",
}

for pkg_name, import_name in _packages.items():
    try:
        __import__(import_name)
        print(f"  [OK]  {pkg_name:24s} already installed")
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pkg_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        print(f"  [OK]  {pkg_name:24s} installed")

print("  All dependencies ready.\n")


# ---------------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------------

import os, io, warnings, tempfile, shutil, requests
import numpy as np

from pathlib import Path
from pdbfixer import PDBFixer
from openmm.app import PDBFile, ForceField, Simulation
from openmm import LangevinMiddleIntegrator, CustomExternalForce, unit
from Bio.PDB import PDBParser, MMCIFParser, PDBIO, Select
from Bio.PDB.Polypeptide import is_aa

RAW_STRUCTURE = None
RAW_STRUCTURE_SOURCE = None

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------------
#  Helper Utilities
# ---------------------------------------------------------------------------

INTERMEDIATE_DIR = Path("/content/prep_intermediates")
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

def _print_header(step_num, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step_num} | {title}")
    print("=" * 72)

def _assign_safe_chain_ids(structure):
    """Map chain IDs to single characters so the structure can be written as valid PDB."""
    chain_pool = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")
    mapping = {}
    for model in structure:
        used = set()
        next_idx = 0
        for chain in list(model.get_chains()):
            old = str(chain.id).strip() if chain.id is not None else ""
            if len(old) == 1 and old not in used:
                new = old
            else:
                while next_idx < len(chain_pool) and chain_pool[next_idx] in used:
                    next_idx += 1
                if next_idx >= len(chain_pool):
                    raise ValueError("Too many chains to remap into PDB single-character chain IDs.")
                new = chain_pool[next_idx]
                next_idx += 1
            chain.id = new
            mapping[old if old else "(blank)"] = new
            used.add(new)
    return mapping


def _renumber_residues_for_pdb(structure):
    """Renumber residues sequentially per chain so downstream PDB parsers stay stable."""
    for model in structure:
        for chain in model:
            new_id = 1
            for residue in list(chain.get_residues()):
                hetflag, _, _ = residue.id
                residue.id = (hetflag, new_id, " ")
                new_id += 1


def _sanitize_structure_for_pdb(structure):
    chain_map = _assign_safe_chain_ids(structure)
    _renumber_residues_for_pdb(structure)
    for atom in structure.get_atoms():
        if atom.get_altloc() == ".":
            atom.set_altloc(" ")
    return chain_map


def _write_structure_to_pdb(structure, out_pdb):
    io_pdb = PDBIO()
    io_pdb.set_structure(structure)
    io_pdb.save(out_pdb)


def _fetch_structure_with_fallback(pdb_code: str, out_pdb: str):
    """Try RCSB PDB first; if unavailable, fetch CIF, sanitize, and write a PDB-safe copy."""
    global RAW_STRUCTURE, RAW_STRUCTURE_SOURCE

    pdb_code = pdb_code.strip().upper()

    pdb_url = f"https://files.rcsb.org/download/{pdb_code}.pdb"
    r = requests.get(pdb_url, timeout=60)
    if r.status_code == 200 and "ATOM" in r.text:
        with open(out_pdb, "w") as f:
            f.write(r.text)
        RAW_STRUCTURE = PDBParser(QUIET=True).get_structure(pdb_code, out_pdb)
        RAW_STRUCTURE_SOURCE = "pdb"
        print(f"  Fetched {pdb_code} as PDB -> {out_pdb}")
        return out_pdb

    cif_url = f"https://files.rcsb.org/download/{pdb_code}.cif"
    r = requests.get(cif_url, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Could not fetch {pdb_code} from RCSB as PDB or CIF.")

    cif_path = str(INTERMEDIATE_DIR / f"{pdb_code}.cif")
    with open(cif_path, "w") as f:
        f.write(r.text)
    print(f"  Fetched {pdb_code} as mmCIF -> {cif_path}")

    parser_cif = MMCIFParser(QUIET=True)
    RAW_STRUCTURE = parser_cif.get_structure(pdb_code, cif_path)
    chain_map = _sanitize_structure_for_pdb(RAW_STRUCTURE)
    _write_structure_to_pdb(RAW_STRUCTURE, out_pdb)
    RAW_STRUCTURE_SOURCE = "cif"
    print(f"  Converted mmCIF to PDB-safe file -> {out_pdb}")
    if chain_map:
        print(f"  PDB-safe chain remap applied -> {chain_map}")
    return out_pdb

def _atom_count(pdb_path):
    count = 0
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                count += 1
    return count

def _infer_element(atom_name: str) -> str:
    name = atom_name.strip()
    letters = "".join([c for c in name if c.isalpha()])
    if not letters:
        return "C"
    if len(letters) >= 2 and letters[:2].title() in {
        "Cl", "Br", "Na", "Mg", "Zn", "Fe", "Ca", "Mn", "Cu", "Co", "Ni"
    }:
        return letters[:2].title()
    return letters[0].upper()


def _format_atom_name(atom_name: str, element: str) -> str:
    name = atom_name.strip()[:4]
    element = (element or "").strip()
    if len(name) >= 4:
        return name[:4]
    if len(element) == 1:
        return f" {name:<3s}"
    return f"{name:>4s}"


def _write_residue_pdb_fixed(residue, chain_id: str, out_pdb: str, resname_override: str = "LIG"):
    lines = []
    serial = 1
    resseq = 1
    resname = (resname_override or residue.get_resname().strip()[:3] or "LIG")[:3]
    for atom in residue.get_atoms():
        coord = atom.get_coord()
        occ = atom.get_occupancy() if atom.get_occupancy() is not None else 1.00
        bfac = atom.get_bfactor() if atom.get_bfactor() is not None else 0.00
        element = getattr(atom, "element", None) or _infer_element(atom.get_name())
        atom_name = _format_atom_name(atom.get_name(), element)
        charge = ""
        line = (
            f"HETATM{serial:5d} {atom_name}{' ':1s}{resname:>3s} {chain_id:1s}{resseq:4d}{' ':1s}   "
            f"{coord[0]:8.3f}{coord[1]:8.3f}{coord[2]:8.3f}"
            f"{occ:6.2f}{bfac:6.2f}          {element:>2s}{charge:>2s}"
        )
        lines.append(line)
        serial += 1
    lines.append("END")
    with open(out_pdb, "w") as fh:
        fh.write("\n".join(lines) + "\n")


def _is_protein_like_residue(residue) -> bool:
    if residue.id[0] == "W":
        return False
    return is_aa(residue, standard=False)

class SingleResidueSelect(Select):
    def __init__(self, target_chain, hetflag, resseq, icode):
        self.target_chain = target_chain
        self.target_id = (hetflag, resseq, icode)
    def accept_chain(self, chain):
        return chain.id == self.target_chain
    def accept_residue(self, residue):
        return residue.id == self.target_id

# ---------------------------------------------------------------------------
#  STEP 1 : Protein Input
# ---------------------------------------------------------------------------

_print_header(1, "Protein Input")

if structure_source == "Upload PDB file":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded. Please re-run and select a PDB file.")
    upload_name = list(uploaded.keys())[0]
    raw_path = str(INTERMEDIATE_DIR / "01_raw_input.pdb")
    with open(raw_path, "wb") as f:
        f.write(uploaded[upload_name])
    RAW_STRUCTURE = PDBParser(QUIET=True).get_structure("uploaded", raw_path)
    RAW_STRUCTURE_SOURCE = "pdb"
    print(f"  Uploaded '{upload_name}' -> {raw_path}")

elif structure_source == "Fetch from RCSB PDB":
    if not pdb_id.strip():
        raise ValueError("Please enter a PDB accession ID in the pdb_id parameter.")
    raw_path = str(INTERMEDIATE_DIR / "01_raw_input.pdb")
    _fetch_structure_with_fallback(pdb_id, raw_path)

else:
    raise ValueError("Invalid structure_source selection.")


# ---------------------------------------------------------------------------
#  STEP 2 : Initial Structure Audit + Reference Ligand Extraction
# ---------------------------------------------------------------------------

_print_header(2, "Initial Structure Audit + Reference Ligand Extraction")

parser = PDBParser(QUIET=True)
if RAW_STRUCTURE is not None:
    structure = RAW_STRUCTURE
    print(f"  Using in-memory structure parsed from {RAW_STRUCTURE_SOURCE.upper()}")
else:
    structure = parser.get_structure("input", raw_path)

models       = list(structure.get_models())
chains       = [c.id for c in models[0].get_chains()]
residues     = [r for r in models[0].get_residues() if r.id[0] == " "]
hetatm_res   = [r for r in models[0].get_residues() if r.id[0] not in (" ", "W")]
waters       = [r for r in models[0].get_residues() if r.id[0] == "W"]
atoms        = list(models[0].get_atoms())
altloc_atoms = [a for a in atoms if a.get_altloc() not in (" ", "", "A")]
ins_code_res = [r for r in models[0].get_residues() if r.id[2].strip()]

print(f"  {'Models':<40s} {len(models)}")
print(f"  {'Chains':<40s} {chains}")
print(f"  {'Standard AA residues':<40s} {len(residues)}")
print(f"  {'HETATM residues':<40s} {len(hetatm_res)}")
print(f"  {'Water molecules':<40s} {len(waters)}")
print(f"  {'Total atoms':<40s} {len(atoms)}")
print(f"  {'Alt-conformation atoms':<40s} {len(altloc_atoms)}")
print(f"  {'Residues with insertion codes':<40s} {len(ins_code_res)}")

keep_cofactors = {x.strip().upper() for x in cofactor_names_csv.split(",") if x.strip()}
candidate_refs = []
for chain in models[0]:
    for residue in chain.get_residues():
        hetflag, resseq, icode = residue.id
        resname = residue.get_resname().strip().upper()
        if hetflag in (" ", "W"):
            continue
        if resname in keep_cofactors:
            continue
        if len(list(residue.get_atoms())) < 5:
            continue
        candidate_refs.append({
            "chain": chain.id,
            "residue": residue,
            "resname": resname,
            "atom_count": len(list(residue.get_atoms()))
        })

reference_ligand_path = None
reference_ligand_name = None

if candidate_refs:
    if reference_ligand_resname.strip().lower() != "auto":
        wanted = reference_ligand_resname.strip().upper()
        filtered = [x for x in candidate_refs if x["resname"] == wanted]
        if filtered:
            candidate_refs = filtered
        else:
            print(f"  WARNING: Requested reference ligand '{wanted}' not found; using auto selection.")

    candidate_refs = sorted(candidate_refs, key=lambda x: x["atom_count"], reverse=True)
    chosen = candidate_refs[0]
    res = chosen["residue"]
    chain_id = chosen["chain"]
    reference_ligand_name = chosen["resname"]

    ref_path = str(INTERMEDIATE_DIR / f"02_reference_ligand_{reference_ligand_name}.pdb")
    _write_residue_pdb_fixed(res, chain_id, ref_path, resname_override="LIG")
    reference_ligand_path = ref_path

    print("\n  Reference ligand candidates (non-water, excluding cofactors):")
    for c in candidate_refs[:10]:
        print(f"    {c['resname']:>4s} | chain {c['chain']} | atoms {c['atom_count']:>3d}")

    print(f"\n  Selected reference ligand -> {reference_ligand_name}")
    print(f"  Saved reference ligand -> {reference_ligand_path}")
else:
    print("\n  WARNING: No suitable reference ligand candidate was auto-detected.")
    print("  You can still dock later using manual / blind / residue-guided box definition.")

# Disulfide scan
print("\n  Scanning for disulfide bonds (CYS SG distance < 2.5 A) ...")
cys_sg = [
    (r, a) for r in residues if r.get_resname() == "CYS"
    for a in r.get_atoms() if a.get_name() == "SG"
]
disulfides = []
for i in range(len(cys_sg)):
    for j in range(i + 1, len(cys_sg)):
        d = cys_sg[i][1] - cys_sg[j][1]
        if d < 2.5:
            disulfides.append((cys_sg[i][0], cys_sg[j][0], d))
if disulfides:
    for r1, r2, d in disulfides:
        print(f"    Disulfide: {r1.get_resname()} {r1.id[1]} -- {r2.get_resname()} {r2.id[1]}  ({d:.2f} A)")
else:
    print("  No disulfide bonds detected.")


# ---------------------------------------------------------------------------
#  STEP 3 : Chain Selection
# ---------------------------------------------------------------------------

_print_header(3, "Chain Selection")

print(f"  Chains available: {chains}")

if len(chains) == 1:
    selected_chain = chains[0]
    print(f"  Single chain '{selected_chain}' -- no selection needed.")
else:
    selected_chain = chains
    print(f"  Multiple chains found. Keeping all: {selected_chain}")
    print("  (Edit 'selected_chain' in this block to restrict if needed.)")

class ChainSelect(Select):
    def __init__(self, chain_ids):
        self.chain_ids = chain_ids if isinstance(chain_ids, list) else [chain_ids]
    def accept_chain(self, chain):
        return chain.id in self.chain_ids
    def accept_residue(self, residue):
        return _is_protein_like_residue(residue)

chain_out = str(INTERMEDIATE_DIR / "03_chain_selected.pdb")
io_pdb = PDBIO()
io_pdb.set_structure(structure)
io_pdb.save(chain_out, ChainSelect(selected_chain))
print(f"  Chain-selected protein-only structure -> {chain_out}")


# ---------------------------------------------------------------------------
#  STEP 4 : PDBFixer -- Gaps, Missing Heavy Atoms, APO Conversion
# ---------------------------------------------------------------------------

_print_header(4, "PDBFixer: Gaps, Missing Heavy Atoms, APO Conversion")

fixer = PDBFixer(filename=chain_out)

fixer.findMissingResidues()
n_gaps = sum(len(v) for v in fixer.missingResidues.values())
if n_gaps > 0:
    print(f"  Found {n_gaps} missing residue(s) in sequence gaps -- adding ...")
    fixer.findMissingResidues()
else:
    print("  No sequence gaps.")

fixer.findNonstandardResidues()
if fixer.nonstandardResidues:
    print(f"  Replacing {len(fixer.nonstandardResidues)} non-standard residue(s) ...")
    fixer.replaceNonstandardResidues()
else:
    print("  No non-standard residues.")

print("  Removing all heteroatoms and water (safety step; chain_selected.pdb is already protein-only) ...")
fixer.removeHeterogens(True)

fixer.findMissingAtoms()
n_missing = sum(len(v) for v in fixer.missingAtoms.values())
n_terminal = sum(len(v) for v in fixer.missingTerminals.values())
print(f"  Missing heavy atoms: {n_missing}  |  Missing terminal atoms: {n_terminal}")
fixer.addMissingAtoms()
print("  All missing heavy atoms added.")

print("  Hydrogens deferred to pdb2pqr for pH-aware placement.")

apo_path = str(INTERMEDIATE_DIR / "04_pdbfixer_apo.pdb")
PDBFile.writeFile(fixer.topology, fixer.positions, open(apo_path, "w"))
print(f"  PDBFixer output -> {apo_path}")


# ---------------------------------------------------------------------------
#  STEP 5 : BioPython cleanup
# ---------------------------------------------------------------------------

_print_header(5, "BioPython: Alt-Conformers, Insertion Codes, Single Model")

structure = parser.get_structure("apo", apo_path)

n_altloc_fixed = 0
for atom in structure.get_atoms():
    if atom.get_altloc() not in (" ", ""):
        atom.set_altloc(" ")
        atom.set_occupancy(1.0)
        n_altloc_fixed += 1
print(f"  Normalised {n_altloc_fixed} altloc atoms -> blank (occupancy 1.0).")

n_renumbered = 0
for model in structure:
    for chain in model:
        new_id = 1
        for residue in list(chain.get_residues()):
            old_id = residue.id
            if old_id[1] != new_id or old_id[2] != " ":
                residue.id = (old_id[0], new_id, " ")
                n_renumbered += 1
            new_id += 1
if n_renumbered > 0:
    print(f"  Renumbered {n_renumbered} residue(s) to sequential IDs (required by pdb2pqr).")
else:
    print("  Residue numbering already sequential.")

cleaned_path = str(INTERMEDIATE_DIR / "05_biopy_cleaned.pdb")
io_pdb.set_structure(structure)
io_pdb.save(cleaned_path)
n_res = sum(1 for r in structure.get_residues() if r.id[0] == " ")
print(f"  Cleaned structure -> {cleaned_path}  ({n_res} residues)")


# ---------------------------------------------------------------------------
#  STEP 6 : PropKa
# ---------------------------------------------------------------------------

_print_header(6, "PropKa: Per-Residue pKa Prediction")

propka_report_path = str(INTERMEDIATE_DIR / "06_propka_report.pka")

try:
    from propka.run import single as propka_single
    propka_mol = propka_single(cleaned_path, optargs=["--pH", str(target_pH)])
    with open(propka_report_path, "w") as pka_out:
        propka_mol.write_pka(pka_out)
    print(f"  PropKa report -> {propka_report_path}")
except Exception as e:
    print(f"  WARNING: PropKa standalone run failed ({e}).")
    print("  pdb2pqr will run PropKa internally in Step 7.")


# ---------------------------------------------------------------------------
#  STEP 7 : pdb2pqr
# ---------------------------------------------------------------------------

_print_header(7, "pdb2pqr: Protonation + H-Bond Network Optimisation")

pqr_out   = str(INTERMEDIATE_DIR / "07_protonated_hbond.pqr")
pdb_h_out = str(INTERMEDIATE_DIR / "07_protonated_hbond.pdb")

pdb2pqr_cmd = [
    sys.executable, "-m", "pdb2pqr",
    "--ff", "AMBER",
    "--ffout", "AMBER",
    "--with-ph", str(target_pH),
    "--titration-state-method", "propka",
    "--pdb-output", pdb_h_out,
    cleaned_path,
    pqr_out,
]

result = subprocess.run(pdb2pqr_cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("  WARNING: pdb2pqr debumper failed -- retrying with --nodebump ...")
    pdb2pqr_cmd_nodebump = [
        sys.executable, "-m", "pdb2pqr",
        "--ff", "AMBER",
        "--ffout", "AMBER",
        "--with-ph", str(target_pH),
        "--titration-state-method", "propka",
        "--nodebump",
        "--pdb-output", pdb_h_out,
        cleaned_path,
        pqr_out,
    ]
    result = subprocess.run(pdb2pqr_cmd_nodebump, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("pdb2pqr failed.")

print("  pdb2pqr completed successfully.")

n_H = 0
with open(pdb_h_out) as f:
    for line in f:
        if line.startswith("ATOM") and line[76:78].strip() == "H":
            n_H += 1
print(f"  Total hydrogens placed: {n_H}")

unmin_path = str(INTERMEDIATE_DIR / "08_unminimised.pdb")
shutil.copy2(pdb_h_out, unmin_path)
print(f"  Unminimised protonated structure -> {unmin_path}")


# ---------------------------------------------------------------------------
#  STEP 8 : OpenMM minimisation
# ---------------------------------------------------------------------------

_print_header(8, "OpenMM: Restrained Energy Minimisation (AMBER14 + GBN2)")

print(f"  Force field  : AMBER14-all + implicit/gbn2")
print(f"  Restraint    : k = {restraint_k} kJ/mol/nm^2 on all heavy atoms")
print(f"  Convergence  : {convergence_tol} kJ/mol/nm  |  max {max_iterations} iterations")

pdb_in = PDBFile(pdb_h_out)
ff = ForceField("amber14-all.xml", "implicit/gbn2.xml")
system = ff.createSystem(pdb_in.topology, nonbondedCutoff=1.0 * unit.nanometers)

restraint_force = CustomExternalForce(
    "0.5 * k * ((x - x0)^2 + (y - y0)^2 + (z - z0)^2)"
)
restraint_force.addGlobalParameter("k", restraint_k * unit.kilojoules_per_mole / unit.nanometers**2)
restraint_force.addPerParticleParameter("x0")
restraint_force.addPerParticleParameter("y0")
restraint_force.addPerParticleParameter("z0")

n_restrained = 0
for i, atom in enumerate(pdb_in.topology.atoms()):
    if atom.element.symbol != "H":
        pos = pdb_in.positions[i]
        restraint_force.addParticle(i, [pos.x, pos.y, pos.z])
        n_restrained += 1

system.addForce(restraint_force)
print(f"  Harmonic restraints on {n_restrained} heavy atoms.")

integrator = LangevinMiddleIntegrator(
    300 * unit.kelvin, 1.0 / unit.picoseconds, 0.002 * unit.picoseconds
)
simulation = Simulation(pdb_in.topology, system, integrator)
simulation.context.setPositions(pdb_in.positions)

state_before = simulation.context.getState(getEnergy=True, getPositions=True)
e_before = state_before.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
pos_before = state_before.getPositions(asNumpy=True).value_in_unit(unit.angstroms)

print("  Minimising ...")
import time
t0 = time.time()
simulation.minimizeEnergy(
    tolerance=convergence_tol * unit.kilojoules_per_mole / unit.nanometers,
    maxIterations=max_iterations,
)
wall_time = time.time() - t0

state_after = simulation.context.getState(getEnergy=True, getPositions=True)
e_after = state_after.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
pos_after = state_after.getPositions(asNumpy=True).value_in_unit(unit.angstroms)

heavy_idx = [i for i, a in enumerate(pdb_in.topology.atoms()) if a.element.symbol != "H"]
delta = pos_after[heavy_idx] - pos_before[heavy_idx]
rmsd = np.sqrt(np.mean(np.sum(delta**2, axis=1)))

print(f"  Potential energy before: {e_before:.1f} kJ/mol")
print(f"  Potential energy after : {e_after:.1f} kJ/mol")
print(f"  Heavy-atom RMSD        : {rmsd:.3f} A")
print(f"  Wall time              : {wall_time:.1f} s")

min_path = str(INTERMEDIATE_DIR / "09_minimised.pdb")
PDBFile.writeFile(simulation.topology, state_after.getPositions(), open(min_path, "w"))

final_path = "/content/apo_protein_clean.pdb"
shutil.copy2(min_path, final_path)

print(f"  Final APO output -> {final_path}")


# ---------------------------------------------------------------------------
#  STEP 9 : Final validation
# ---------------------------------------------------------------------------

_print_header(9, "Final Validation and Docking Readiness Checklist")

struct_final = parser.get_structure("final", final_path)
model_final = list(struct_final.get_models())[0]
het_final   = [r for r in model_final.get_residues() if r.id[0] not in (" ", "W")]
wat_final   = [r for r in model_final.get_residues() if r.id[0] == "W"]

print(f"  Remaining HETATM residues : {len(het_final)}")
print(f"  Remaining waters          : {len(wat_final)}")
print(f"  Reference ligand path     : {reference_ligand_path if reference_ligand_path else 'not found'}")
print(f"  APO protein output        : {final_path}")

if len(het_final) == 0 and len(wat_final) == 0:
    print("\n  Protein is READY for Step 1.5 (re-load NADPH) and later docking.")
else:
    print("\n  WARNING: Some HETATM or water records remain. Inspect the output above.")


# ---------------------------------------------------------------------------
#  STEP 10 : Download
# ---------------------------------------------------------------------------

_print_header(10, "Download Key Outputs")

from google.colab import files as colab_files

download_targets = [final_path]
if reference_ligand_path:
    reference_copy = "/content/reference_ligand_from_raw.pdb"
    shutil.copy2(reference_ligand_path, reference_copy)
    download_targets.append(reference_copy)

for fp in download_targets:
    if os.path.exists(fp):
        print(f"  Downloading {os.path.basename(fp)} ...")
        colab_files.download(fp)

print("\n" + "=" * 72)
print("  STEP 1 COMPLETE")
print("=" * 72)
print(f"  APO protein        : {final_path}")
print(f"  Reference ligand   : {reference_ligand_path if reference_ligand_path else 'not extracted'}")
print(f"  Next               : Step 1.5 re-load NADPH into the receptor")
print("=" * 72)


# **Step 1.5: Reload NADPH into the prepared 8RRJ receptor**

This step is inserted between your original Step 1 and Step 2.

It:

- uploads **NADPH.pdb** and optionally **NADPH.mol2**
- rebuilds a docking receptor as **apo protein + NADPH**
- keeps the reference ligand from Step 1 unchanged for later box definition
- prepares a receptor file that can later be converted to **PDBQT with partial charges**


In [ ]:

# =============================================================================
#  Step 1.5 -- Reload NADPH into the prepared receptor
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

merged_receptor_name = "8RRJ_receptor_with_NADPH"  # @param {type:"string"}


# ---------------------------------------------------------------------------
#  STEP 0 : Install / Imports
# ---------------------------------------------------------------------------

import os, sys, shutil, subprocess, re
from pathlib import Path

print("=" * 72)
print("  STEP 0 | Installing Dependencies")
print("=" * 72)

def _run(cmd):
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

try:
    import openbabel
    print(f"  [OK]  {'openbabel':24s} already installed")
except ImportError:
    _run(["apt-get", "install", "-y", "-q", "openbabel"])
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openbabel-wheel"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    print(f"  [OK]  {'openbabel':24s} installed")

print("  Dependencies ready.\n")

from google.colab import files
from openbabel import openbabel as ob


# ---------------------------------------------------------------------------
#  Helpers
# ---------------------------------------------------------------------------

WORKDIR = Path("/content/nadph_reload")
WORKDIR.mkdir(exist_ok=True)

def _print_header(step_num, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step_num} | {title}")
    print("=" * 72)

def _pdb_records_only(path):
    keep = []
    with open(path) as fh:
        for line in fh:
            if line.startswith(("ATOM", "HETATM", "TER")):
                keep.append(line.rstrip("\n"))
    return keep

def _write_merged_pdb(protein_pdb, nadph_pdb, out_pdb):
    prot_lines = _pdb_records_only(protein_pdb)
    nad_lines  = _pdb_records_only(nadph_pdb)

    merged = []
    serial = 1

    for line in prot_lines + nad_lines:
        if line.startswith(("ATOM", "HETATM")):
            atom_name = line[12:16]
            resname   = line[17:20]
            chain_id  = line[21:22] if len(line) >= 22 else "A"
            resseq    = line[22:26] if len(line) >= 26 else "   1"
            x         = float(line[30:38])
            y         = float(line[38:46])
            z         = float(line[46:54])
            occ       = float(line[54:60]) if len(line) >= 60 and line[54:60].strip() else 1.00
            bfac      = float(line[60:66]) if len(line) >= 66 and line[60:66].strip() else 0.00
            element   = "".join([c for c in atom_name if c.isalpha()]).strip()
            if len(element) >= 2 and element[:2].title() in {"Cl","Br","Na","Mg","Zn","Fe","Ca","Mn","Cu","Co","Ni"}:
                element = element[:2].title()
            else:
                element = element[:1].upper() if element else "C"
            record = line[:6].strip() or "HETATM"
            merged.append(
                f"{record:<6}{serial:>5d} {atom_name:<4}{' ':1}{resname:>3s} {chain_id:1s}{int(resseq):>4d}    "
                f"{x:>8.3f}{y:>8.3f}{z:>8.3f}{occ:>6.2f}{bfac:>6.2f}          {element:>2s}"
            )
            serial += 1
        elif line.startswith("TER"):
            merged.append("TER")
    merged.append("END")

    with open(out_pdb, "w") as f:
        f.write("\n".join(merged) + "\n")

def _ob_mol2_to_pdb(mol2_path, pdb_path):
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("mol2", "pdb")
    mol = ob.OBMol()
    if not obc.ReadFile(mol, str(mol2_path)):
        raise RuntimeError(f"OpenBabel could not read {mol2_path}")
    obc.WriteFile(mol, str(pdb_path))

def _mol2_charge_sum(mol2_path):
    total = 0.0
    in_atoms = False
    with open(mol2_path) as fh:
        for line in fh:
            s = line.strip()
            if s == "@<TRIPOS>ATOM":
                in_atoms = True
                continue
            if s.startswith("@<TRIPOS>") and in_atoms:
                break
            if in_atoms and s:
                parts = s.split()
                if len(parts) >= 9:
                    try:
                        total += float(parts[-1])
                    except Exception:
                        pass
    return total


# ---------------------------------------------------------------------------
#  STEP 1 : Check protein output from Step 1
# ---------------------------------------------------------------------------

_print_header(1, "Locate APO protein from Step 1")

protein_pdb = Path("/content/apo_protein_clean.pdb")
if not protein_pdb.exists():
    raise FileNotFoundError("Cannot find /content/apo_protein_clean.pdb. Run Step 1 first.")
print(f"  APO protein found -> {protein_pdb}")


# ---------------------------------------------------------------------------
#  STEP 2 : Upload NADPH
# ---------------------------------------------------------------------------

_print_header(2, "Upload NADPH files")

print("  Please upload NADPH.pdb and, if available, NADPH.mol2 in the same upload window.")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No NADPH files uploaded.")

nadph_pdb = None
nadph_mol2 = None
for name in uploaded.keys():
    lower = name.lower()
    if lower.endswith(".pdb"):
        nadph_pdb = WORKDIR / "NADPH_uploaded.pdb"
        shutil.copy(name, nadph_pdb)
    elif lower.endswith(".mol2"):
        nadph_mol2 = WORKDIR / "NADPH_uploaded.mol2"
        shutil.copy(name, nadph_mol2)

if nadph_pdb is None and nadph_mol2 is None:
    raise RuntimeError("Please upload at least NADPH.pdb or NADPH.mol2.")

if nadph_pdb is None and nadph_mol2 is not None:
    nadph_pdb = WORKDIR / "NADPH_from_mol2.pdb"
    _ob_mol2_to_pdb(nadph_mol2, nadph_pdb)
    print("  NADPH PDB generated from MOL2.")

if nadph_mol2 is not None:
    print(f"  NADPH MOL2 uploaded -> {nadph_mol2}")
    print(f"  NADPH MOL2 net charge (sum of atom charges) = {_mol2_charge_sum(nadph_mol2):.3f}")

print(f"  NADPH PDB ready -> {nadph_pdb}")


# ---------------------------------------------------------------------------
#  STEP 3 : Merge APO protein + NADPH
# ---------------------------------------------------------------------------

_print_header(3, "Merge APO protein with NADPH")

merged_name = re.sub(r"[^\w\-]", "_", merged_receptor_name).strip() or "8RRJ_receptor_with_NADPH"
merged_pdb = Path(f"/content/{merged_name}.pdb")
_write_merged_pdb(str(protein_pdb), str(nadph_pdb), str(merged_pdb))

print(f"  Merged receptor written -> {merged_pdb}")
print(f"  Reference ligand from Step 1 remains available as /content/reference_ligand_from_raw.pdb (if extracted)")


# ---------------------------------------------------------------------------
#  STEP 4 : Download merged receptor
# ---------------------------------------------------------------------------

_print_header(4, "Download merged receptor")

from google.colab import files as colab_files
colab_files.download(str(merged_pdb))

print("\n" + "=" * 72)
print("  STEP 1.5 COMPLETE")
print("=" * 72)
print(f"  Receptor for docking : {merged_pdb}")
print(f"  Next                 : Step 2 ligand preparation from merge_ROCS0.918_EON0.9136.csv")
print("=" * 72)


# **Step 2: Ligand Preparation from `merge_ROCS0.918_EON0.9136.csv` (first column = SMILES)**

This keeps the original ligand-preparation logic as much as possible, but changes the input mode to:

- upload **`merge_ROCS0.918_EON0.9136.csv`**
- use the **first column** as SMILES
- generate 3D conformers, protonate, minimise, and export a multi-molecule **SDF**
- preserve useful identifiers like `Catalog_ID` / `Title` when present


In [ ]:

# =============================================================================
#  Ligand Preparation Pipeline from CSV SMILES
# =============================================================================
#  Adapted from the original notebook with minimal workflow changes:
#    - input = CSV
#    - first column = SMILES
#    - keep the same sanitise -> standardise -> protonate -> embed -> MMFF flow
# =============================================================================

# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

output_filename = "merge_ROCS0.918_EON0.9136"  # @param {type:"string"}
protonation_method = "OpenBabel (pH-aware)"  # @param ["OpenBabel (pH-aware)", "RDKit (neutralise charges)"]
target_pH = 7.4  # @param {type:"number"}
n_conformers = 1  # @param {type:"integer"}
mmff_max_iters = 2000  # @param {type:"integer"}
max_ligands = 0  # @param {type:"integer"}
output_sdf = True  # @param {type:"boolean"}


# ---------------------------------------------------------------------------
#  Resolve parameter values
# ---------------------------------------------------------------------------

OUTPUT_BASENAME    = output_filename.strip().replace(" ", "_") + "_clean"
PROTONATION_METHOD = "openbabel" if "OpenBabel" in protonation_method else "rdkit"
TARGET_PH          = target_pH
N_CONFORMERS       = n_conformers
MMFF_MAX_ITERS     = mmff_max_iters
MAX_LIGANDS        = max_ligands if max_ligands > 0 else None
OUT_SDF            = output_sdf

if not OUT_SDF:
    raise ValueError("For this workflow, please keep output_sdf = True.")


# ---------------------------------------------------------------------------
#  STEP 0 : Install Dependencies
# ---------------------------------------------------------------------------

import sys, subprocess

def _print_header(step_num, title):
    print("\n" + "=" * 72)
    print(f"  STEP {step_num} | {title}")
    print("=" * 72)

print("=" * 72)
print("  STEP 0 | Installing Dependencies")
print("=" * 72)

def _run(cmd):
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for label, pip_name, import_name in [
    ("rdkit",     "rdkit",     "rdkit.Chem"),
    ("tqdm",      "tqdm",      "tqdm"),
    ("pandas",    "pandas",    "pandas"),
]:
    try:
        __import__(import_name)
        print(f"  [OK]  {label:24s} already installed")
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        print(f"  [OK]  {label:24s} installed")

try:
    import openbabel
    print(f"  [OK]  {'openbabel':24s} already installed")
except ImportError:
    _run(["apt-get", "install", "-y", "-q", "openbabel"])
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openbabel-wheel"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    print(f"  [OK]  {'openbabel':24s} installed")

print("  All dependencies ready.\n")


# ---------------------------------------------------------------------------
#  Imports
# ---------------------------------------------------------------------------

import os, time, warnings, tempfile, re
from copy import deepcopy
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdDistGeom
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import SDWriter
from openbabel import openbabel as ob

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------------
#  STEP 1 : File Upload
# ---------------------------------------------------------------------------

_print_header(1, "Upload CSV")

from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise SystemExit("No CSV uploaded. Please re-run the cell.")

upload_name = list(uploaded.keys())[0]
upload_path = Path(upload_name)

if upload_path.suffix.lower() != ".csv":
    raise ValueError(f"Please upload a CSV file. Got '{upload_path.suffix}'.")

print(f"  Uploaded : {upload_name}")
df = pd.read_csv(upload_path)
print(f"  Rows      : {len(df)}")
print(f"  Columns   : {list(df.columns)}")

smiles_col = df.columns[0]
print(f"  Using first column as SMILES -> {smiles_col}")

if MAX_LIGANDS is not None:
    df = df.iloc[:MAX_LIGANDS].copy()
    print(f"  Truncated to first {len(df)} ligands (MAX_LIGANDS)")


# ---------------------------------------------------------------------------
#  STEP 2 : Convert CSV rows to molecules
# ---------------------------------------------------------------------------

_print_header(2, "CSV Parsing")

def _make_ligand_name(row, idx):
    for key in ["Catalog_ID", "Title", "VIDA Name", "Rank"]:
        if key in row and pd.notna(row[key]) and str(row[key]).strip():
            return re.sub(r"[^\w\-]", "_", str(row[key]).strip())
    return f"lig_{idx+1:05d}"

raw_mols = []
failed_rows = []

for idx, row in df.iterrows():
    smi = str(row[smiles_col]).strip()
    if not smi or smi.lower() == "nan":
        failed_rows.append((idx + 1, "empty SMILES"))
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        failed_rows.append((idx + 1, f"invalid SMILES: {smi[:60]}"))
        continue
    name = _make_ligand_name(row, idx)
    mol.SetProp("_Name", name)
    mol.SetProp("Source_SMILES", smi)
    for col in df.columns:
        val = row[col]
        if pd.notna(val):
            mol.SetProp(str(col), str(val))
    raw_mols.append((mol, name))

print(f"  Valid SMILES rows : {len(raw_mols)}")
print(f"  Failed rows       : {len(failed_rows)}")
if failed_rows[:10]:
    print("  First failed rows:")
    for x in failed_rows[:10]:
        print(f"    row {x[0]} -> {x[1]}")


# ---------------------------------------------------------------------------
#  STEP 3 : Ligand Preparation
# ---------------------------------------------------------------------------

_print_header(3, "Ligand Preparation")

def standardise_mol(mol):
    lfc = rdMolStandardize.LargestFragmentChooser()
    te  = rdMolStandardize.TautomerEnumerator()
    mol = lfc.choose(mol)
    mol = te.Canonicalize(mol)
    return mol

def protonate_rdkit(mol):
    uc = rdMolStandardize.Uncharger()
    return uc.uncharge(mol)

def protonate_openbabel(mol, ph=7.4):
    try:
        smi    = Chem.MolToSmiles(mol)
        obconv = ob.OBConversion()
        obconv.SetInAndOutFormats("smi", "smi")
        obmol  = ob.OBMol()
        obconv.ReadString(obmol, smi)
        obmol.AddHydrogens(False, True, ph)
        out_smi = obconv.WriteString(obmol).strip()
        out_mol = Chem.MolFromSmiles(out_smi)
        if out_mol is None:
            return mol
        return out_mol
    except Exception:
        return mol

def embed_and_minimise(mol, n_confs, max_iters):
    params = rdDistGeom.ETKDGv3()
    params.randomSeed            = 42
    params.numThreads            = 0
    params.useSmallRingTorsions  = True
    params.useMacrocycleTorsions = True
    params.enforceChirality      = True

    mol3d = Chem.AddHs(mol)
    conf_ids = AllChem.EmbedMultipleConfs(mol3d, numConfs=n_confs, params=params)
    if not conf_ids:
        AllChem.EmbedMolecule(mol3d, AllChem.ETKDG())

    results = AllChem.MMFFOptimizeMoleculeConfs(
        mol3d, mmffVariant="MMFF94s", maxIters=max_iters
    )
    if not results:
        return mol3d, False

    energies = [(r[1], i) for i, r in enumerate(results) if r[0] == 0]
    if not energies:
        converged, best_conf = False, 0
    else:
        converged = True
        _, best_conf = min(energies)

    mol_out = deepcopy(mol3d)
    for cid in reversed(range(mol_out.GetNumConformers())):
        if cid != best_conf:
            mol_out.RemoveConformer(cid)
    return mol_out, converged

prepared = []
failed = []

for idx, (raw_mol, mol_name) in enumerate(
        tqdm(raw_mols, desc="  Preparing", unit="mol", bar_format="{l_bar}{bar:30}{r_bar}")):
    try:
        mol = Chem.Mol(raw_mol)
        Chem.SanitizeMol(mol)
        mol = standardise_mol(mol)

        if PROTONATION_METHOD == "openbabel":
            mol = protonate_openbabel(mol, ph=TARGET_PH)
        else:
            mol = protonate_rdkit(mol)

        mol3d, converged = embed_and_minimise(mol, n_confs=N_CONFORMERS, max_iters=MMFF_MAX_ITERS)
        if mol3d.GetNumConformers() == 0:
            raise ValueError("3D embedding failed -- no conformer generated")

        # restore props
        for k, v in raw_mol.GetPropsAsDict().items():
            mol3d.SetProp(str(k), str(v))
        mol3d.SetProp("_Name", mol_name)
        mol3d.SetProp("Converged", str(converged))
        prepared.append((mol3d, mol_name))

    except Exception as err:
        failed.append((idx + 1, mol_name, str(err)))

print(f"  Prepared successfully : {len(prepared)}")
print(f"  Failed                : {len(failed)}")
if failed[:10]:
    print("  First failed ligands:")
    for item in failed[:10]:
        print(f"    {item}")


# ---------------------------------------------------------------------------
#  STEP 4 : Writing Output SDF
# ---------------------------------------------------------------------------

_print_header(4, "Writing Output SDF")

out_sdf = Path(f"/content/{OUTPUT_BASENAME}.sdf")
with SDWriter(str(out_sdf)) as w:
    w.SetKekulize(False)
    for mol, name in prepared:
        w.write(mol)

print(f"  [OK]  SDF -> {out_sdf}  ({len(prepared)} ligands)")


# ---------------------------------------------------------------------------
#  STEP 5 : Download
# ---------------------------------------------------------------------------

_print_header(5, "Download Prepared Ligands")

from google.colab import files as colab_files
colab_files.download(str(out_sdf))

print("\n" + "=" * 72)
print("  STEP 2 COMPLETE")
print("=" * 72)
print(f"  Prepared ligand SDF : {out_sdf}")
print(f"  Next                : Step 4 docking with receptor + NADPH")
print("=" * 72)


# **Step 4: Molecular Docking Run (8RRJ receptor + reloaded NADPH + prepared ligands)**

This docking cell keeps the original Vina/OpenBabel structure as much as possible, but now:

- uses the receptor produced by **Step 1.5**
- uses the ligand SDF produced by **Step 2**
- uses the **reference ligand extracted from raw 8RRJ** to define the docking box automatically
- audits **partial charges** in generated PDBQT files
- exports:
  - per-ligand docking folders
  - ranked docking summary CSV
  - **top 10 final best-pose SDF** (`top10_ranked_best_poses.sdf`)


In [ ]:

# =============================================================================
#  AutoDock Vina -- Docking Pipeline (8RRJ + reloaded NADPH)
# =============================================================================
#  Adapted from the original notebook with minimal structural changes:
#    - no manual file upload here; consume outputs from Steps 1.5 and 2
#    - docking box auto-defined from the Step 1 reference ligand
#    - keep partial-charge checks for receptor/ligands
#    - export ranked top 10 best-pose SDF
# =============================================================================


# ---------------------------------------------------------------------------
#  USER PARAMETERS (Google Colab @param)
# ---------------------------------------------------------------------------

# @markdown ### Binding-Site Definition
BOX_MODE = "D -- Reference ligand"  # @param ["A -- Blind docking", "B -- Manual centre + box", "C -- Residue-guided", "D -- Reference ligand"]

# @markdown #### Mode A -- Blind Docking
BLIND_PADDING = 5.0  # @param {type:"number"}

# @markdown #### Mode B -- Manual Centre and Box Size (Angstroms)
MANUAL_CENTER_X = 0.0   # @param {type:"number"}
MANUAL_CENTER_Y = 0.0   # @param {type:"number"}
MANUAL_CENTER_Z = 0.0   # @param {type:"number"}
MANUAL_SIZE_X   = 25.0  # @param {type:"number"}
MANUAL_SIZE_Y   = 25.0  # @param {type:"number"}
MANUAL_SIZE_Z   = 25.0  # @param {type:"number"}

# @markdown #### Mode C -- Residue-Guided
KEY_RESIDUES_CSV = "45, 89, 120"  # @param {type:"string"}
RESIDUE_PADDING  = 8.0  # @param {type:"number"}

# @markdown #### Mode D -- Reference Ligand
REF_LIGAND_PADDING = 6.0  # @param {type:"number"}

# @markdown ---
# @markdown ### Docking Engine
SCORING_FUNCTION = "vina"  # @param ["vina", "vinardo"]
EXHAUSTIVENESS   = 8       # @param {type:"integer"}
N_POSES          = 10      # @param {type:"integer"}
CPU_CORES        = 0       # @param {type:"integer"}
RANDOM_SEED      = 42      # @param {type:"integer"}

# @markdown ---
# @markdown ### Output Options
RUN_NAME          = "8RRJ_NADPH_docking"  # @param {type:"string"}
AFFINITY_CUTOFF   = "none"  # @param {type:"string"}
EXPORT_POSE_PDBS  = True    # @param {type:"boolean"}
EXPORT_MERGED_SDF = True    # @param {type:"boolean"}
SKIP_FAILED       = True    # @param {type:"boolean"}

# @markdown ---
# @markdown ### Ligand Selection
# @markdown How many ligands to dock? Set to **0** to dock **all**.
LIGAND_LIMIT = 0  # @param {type:"integer"}

# @markdown ---
# @markdown ### Input filenames
RECEPTOR_PDB_NAME = "8RRJ_receptor_with_NADPH.pdb"  # @param {type:"string"}
LIGAND_SDF_NAME   = "merge_ROCS0.918_EON0.9136_clean.sdf"  # @param {type:"string"}


# ---------------------------------------------------------------------------
#  Resolve form values
# ---------------------------------------------------------------------------

import subprocess, sys, os, shutil, time, zipfile, csv, re
from pathlib import Path

box_mode          = BOX_MODE[0]
scoring_fn        = SCORING_FUNCTION.strip().lower()
exhaustiveness    = EXHAUSTIVENESS
n_poses           = N_POSES
cpu_cores         = CPU_CORES
seed              = RANDOM_SEED
run_name          = re.sub(r"[^\w\-]", "_", RUN_NAME) if RUN_NAME else "run1"
affinity_cutoff   = None if AFFINITY_CUTOFF.strip().lower() in ("none", "") else float(AFFINITY_CUTOFF)
export_pose_pdbs  = EXPORT_POSE_PDBS
export_merged_sdf = EXPORT_MERGED_SDF
skip_failed       = SKIP_FAILED
ligand_limit      = LIGAND_LIMIT if LIGAND_LIMIT > 0 else None
key_res           = [int(r.strip()) for r in KEY_RESIDUES_CSV.split(",") if r.strip().isdigit()]
pad_auto          = BLIND_PADDING
pad_res           = RESIDUE_PADDING
pad_ref           = REF_LIGAND_PADDING
center            = None
size              = None
ref_lig_path      = None

if box_mode == "B":
    center = [MANUAL_CENTER_X, MANUAL_CENTER_Y, MANUAL_CENTER_Z]
    size   = [MANUAL_SIZE_X,   MANUAL_SIZE_Y,   MANUAL_SIZE_Z]

if box_mode == "C" and not key_res:
    print("  WARNING: No valid residues in KEY_RESIDUES_CSV -- falling back to blind docking (A)")
    box_mode = "A"


# ---------------------------------------------------------------------------
#  STEP 1 : Install Dependencies
# ---------------------------------------------------------------------------

def _pip(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    return r.returncode == 0

def _print_header(step_label, title):
    print("\n" + "=" * 72)
    print(f"  {step_label} | {title}")
    print("=" * 72)

_print_header("STEP 1/7", "Installing Dependencies")

_PKGS = {
    "vina"            : "AutoDock Vina 1.2.x Python bindings",
    "openbabel-wheel" : "Format conversion (PDB/SDF/MOL2 to PDBQT)",
}
for pkg, desc in _PKGS.items():
    status = "installed" if _pip(pkg) else "FAILED"
    print(f"  [{'OK' if status == 'installed' else '!!'}]  {pkg:<22s} {desc}  ({status})")

try:
    from rdkit import Chem
    print(f"  [OK]  {'rdkit':<22s} already installed")
except ImportError:
    _pip("rdkit-pypi")
    print(f"  [OK]  {'rdkit':<22s} installed")

print("  All dependencies ready.")


# ---------------------------------------------------------------------------
#  STEP 2 : Locate inputs
# ---------------------------------------------------------------------------

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from openbabel import openbabel as ob

WORKDIR = Path("/content/vina_docking_8RRJ")
WORKDIR.mkdir(exist_ok=True)

_print_header("STEP 2/7", "Locate Inputs")

prot_in = Path(f"/content/{RECEPTOR_PDB_NAME}")
lig_in  = Path(f"/content/{LIGAND_SDF_NAME}")
ref_default = Path("/content/reference_ligand_from_raw.pdb")

if not prot_in.exists():
    raise FileNotFoundError(f"Cannot find receptor file: {prot_in}. Run Step 1.5 first.")
if not lig_in.exists():
    raise FileNotFoundError(f"Cannot find ligand file: {lig_in}. Run Step 2 first.")
if box_mode == "D" and not ref_default.exists():
    raise FileNotFoundError("Cannot find /content/reference_ligand_from_raw.pdb. Run Step 1 first.")

ref_lig_path = ref_default if ref_default.exists() else None

print(f"  Receptor : {prot_in}")
print(f"  Ligands  : {lig_in}")
print(f"  Ref lig  : {ref_lig_path if ref_lig_path else 'not available'}")


# ---------------------------------------------------------------------------
#  STEP 3 : Settings Confirmation
# ---------------------------------------------------------------------------

_print_header("STEP 3/7", "Settings Confirmation")

_lim_disp = f"{ligand_limit}" if ligand_limit else "all"
_co_disp  = f"{affinity_cutoff} kcal/mol" if affinity_cutoff is not None else "none"

print(f"""
  Receptor       : {prot_in.name}
  Ligands        : {lig_in.name}
  Site mode      : {box_mode}
  Scoring fn     : {scoring_fn}
  Exhaustiveness : {exhaustiveness}
  Poses          : {n_poses}
  CPUs           : {'auto' if cpu_cores == 0 else cpu_cores}
  Seed           : {seed}
  Run name       : {run_name}
  Cutoff         : {_co_disp}
  Dock           : {_lim_disp} ligand(s)
""")


# ---------------------------------------------------------------------------
#  Helpers
# ---------------------------------------------------------------------------

def _load_mol(filepath):
    ext = filepath.suffix.lower()
    mol = None
    if ext == ".sdf":
        mol = next((m for m in Chem.SDMolSupplier(str(filepath), removeHs=False) if m is not None), None)
    elif ext == ".mol2":
        mol = Chem.MolFromMol2File(str(filepath), removeHs=False)
    elif ext == ".pdb":
        mol = Chem.MolFromPDBFile(str(filepath), removeHs=False)
    if mol is None:
        try:
            oc = ob.OBConversion()
            oc.SetInAndOutFormats(ext.lstrip("."), "sdf")
            om = ob.OBMol()
            oc.ReadFile(om, str(filepath))
            tmp = filepath.parent / f"{filepath.stem}_conv.sdf"
            oc.WriteFile(om, str(tmp))
            mol = next((m for m in Chem.SDMolSupplier(str(tmp), removeHs=False) if m is not None), None)
        except Exception:
            mol = None
    return mol

def _collect_mols(sdf_path):
    mols = []
    for i, mol in enumerate(Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=True)):
        if mol is None:
            continue
        name = mol.GetPropsAsDict().get("_Name", "").strip() or f"mol_{i+1:05d}"
        mol.SetProp("_Name", name)
        mols.append((name, mol))
    return mols

def _ob_sdf_to_pdbqt(sdf_path, pdbqt_path):
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("sdf", "pdbqt")
    mol = ob.OBMol()
    if not obc.ReadFile(mol, str(sdf_path)):
        raise RuntimeError(f"OpenBabel could not read {sdf_path.name}")
    cm2 = ob.OBChargeModel.FindType("gasteiger")
    if cm2:
        cm2.ComputeCharges(mol)
    obc.WriteFile(mol, str(pdbqt_path))

def _pdbqt_to_sdf(pdbqt_path, sdf_path):
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("pdbqt", "sdf")
    mol = ob.OBMol()
    if not obc.ReadFile(mol, str(pdbqt_path)):
        raise RuntimeError(f"OpenBabel could not read {pdbqt_path.name}")
    obc.WriteFile(mol, str(sdf_path))

def _pdbqt_charge_stats(pdbqt_path):
    charges = []
    with open(pdbqt_path) as fh:
        for line in fh:
            if line.startswith(("ATOM", "HETATM")):
                tail = line[66:].split()
                val = None
                if len(tail) >= 1:
                    try:
                        val = float(tail[0])
                    except Exception:
                        val = None
                if val is None:
                    try:
                        val = float(line[70:76])
                    except Exception:
                        val = None
                if val is not None:
                    charges.append(val)
    nonzero = sum(abs(c) > 1e-6 for c in charges)
    return {
        "n_atoms": len(charges),
        "nonzero_atoms": nonzero,
        "sum_charge": float(sum(charges)) if charges else float("nan"),
        "min_charge": float(min(charges)) if charges else float("nan"),
        "max_charge": float(max(charges)) if charges else float("nan"),
    }


# ---------------------------------------------------------------------------
#  STEP 4 : Protein Conversion + Docking Box
# ---------------------------------------------------------------------------

_print_header("STEP 4/7", "Protein Conversion (PDB to PDBQT) + Docking Box")

# -- 4a. Parse protein coordinates ------------------------------------------
all_coords, res_coords = [], {}
with open(prot_in) as fh:
    for line in fh:
        if not line.startswith("ATOM"):
            continue
        try:
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            rnum    = int(line[22:26])
            all_coords.append([x, y, z])
            res_coords.setdefault(rnum, []).append([x, y, z])
        except Exception:
            continue

if not all_coords:
    raise RuntimeError("No ATOM records in receptor PDB.")

coords_np = np.array(all_coords)
prot_min  = coords_np.min(axis=0)
prot_max  = coords_np.max(axis=0)
print(f"  Receptor atoms parsed: {len(all_coords)}")
print(f"  Protein bounds: X=[{prot_min[0]:.1f},{prot_max[0]:.1f}] "
      f"Y=[{prot_min[1]:.1f},{prot_max[1]:.1f}] "
      f"Z=[{prot_min[2]:.1f},{prot_max[2]:.1f}] A")

# -- 4b. Compute docking box ------------------------------------------------
if box_mode == "C":
    res_pts, found, missing = [], [], []
    for r in key_res:
        pts = res_coords.get(r)
        if pts:
            res_pts.extend(pts); found.append(r)
        else:
            missing.append(r)
    if missing:
        print(f"  WARNING: Residues not found: {missing}")
    if res_pts:
        rp = np.array(res_pts)
        center = rp.mean(axis=0).tolist()
        size   = (rp.max(axis=0) - rp.min(axis=0) + 2 * pad_res).clip(min=15).tolist()
    else:
        print("  WARNING: Falling back to blind docking")
        box_mode = "A"

if box_mode == "D" and ref_lig_path:
    ref_mol = _load_mol(ref_lig_path)
    if ref_mol is None or ref_mol.GetNumConformers() == 0:
        print("  WARNING: Reference ligand parsing failed; falling back to blind docking")
        box_mode = "A"
    else:
        ref_pos = np.array([ref_mol.GetConformer().GetAtomPosition(i) for i in range(ref_mol.GetNumAtoms())])
        center  = ref_pos.mean(axis=0).tolist()
        size    = (ref_pos.max(axis=0) - ref_pos.min(axis=0) + 2 * pad_ref).clip(min=15).tolist()
        print(f"  Box from reference ligand -> center=({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}) A")

if box_mode == "A" or center is None or size is None:
    center = ((prot_min + prot_max) / 2.0).tolist()
    size   = ((prot_max - prot_min) + 2 * pad_auto).clip(min=20).tolist()
    print("  Using blind-docking box from whole protein.")

print(f"  Final box center : ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}) A")
print(f"  Final box size   : ({size[0]:.2f}, {size[1]:.2f}, {size[2]:.2f}) A")

# -- 4c. Convert receptor PDB -> PDBQT -------------------------------------
obC = ob.OBConversion()
obC.SetInAndOutFormats("pdb", "pdbqt")
obC.AddOption("r", ob.OBConversion.OUTOPTIONS)

prot_mol = ob.OBMol()
if not obC.ReadFile(prot_mol, str(prot_in)):
    raise RuntimeError("OpenBabel could not read receptor PDB.")

cm = ob.OBChargeModel.FindType("gasteiger")
if cm:
    cm.ComputeCharges(prot_mol)

prot_pdbqt = WORKDIR / "receptor_with_nadph.pdbqt"
obC.WriteFile(prot_mol, str(prot_pdbqt))

rec_charge = _pdbqt_charge_stats(prot_pdbqt)
print("\n  Receptor PDBQT charge audit")
print(f"    atoms with parsed charge : {rec_charge['n_atoms']}")
print(f"    non-zero partial charges : {rec_charge['nonzero_atoms']}")
print(f"    summed charge            : {rec_charge['sum_charge']:.3f}")
print(f"    min / max atom charge    : {rec_charge['min_charge']:.3f} / {rec_charge['max_charge']:.3f}")


# ---------------------------------------------------------------------------
#  STEP 5 : Read ligand set
# ---------------------------------------------------------------------------

_print_header("STEP 5/7", "Read Prepared Ligands")

mol_list = _collect_mols(lig_in)
print(f"  Parsed ligands: {len(mol_list)}")
if not mol_list:
    raise RuntimeError("No valid ligands parsed from prepared SDF.")

if ligand_limit is not None and len(mol_list) > ligand_limit:
    print(f"  Capping to first {ligand_limit} ligands")
    mol_list = mol_list[:ligand_limit]


# ---------------------------------------------------------------------------
#  STEP 6 : Docking Loop
# ---------------------------------------------------------------------------

_print_header("STEP 6/7", f"Docking ({len(mol_list)} ligand(s))")

from vina import Vina

summary_rows = []
best_pose_mols = []
failed_rows = []
t0_total = time.time()

for lig_idx, (lig_name, mol_raw) in enumerate(mol_list):
    lig_num = lig_idx + 1
    print(f"\n  {'-' * 56}")
    print(f"  Ligand {lig_num}/{len(mol_list)}  |  {lig_name}\n")

    try:
        mw      = Descriptors.ExactMolWt(mol_raw)
        logp    = Descriptors.MolLogP(mol_raw)
        hbd     = rdMolDescriptors.CalcNumHBD(mol_raw)
        hba     = rdMolDescriptors.CalcNumHBA(mol_raw)
        rotb    = rdMolDescriptors.CalcNumRotatableBonds(mol_raw)
        tpsa    = Descriptors.TPSA(mol_raw)
        formula = rdMolDescriptors.CalcMolFormula(mol_raw)
        formal_charge = Chem.GetFormalCharge(Chem.RemoveHs(mol_raw))

        print(f"    [prop]  {formula}  MW={mw:.1f}  LogP={logp:.2f}  HBD={hbd}  HBA={hba}  RotB={rotb}  TPSA={tpsa:.0f} A^2")
        print(f"    [prop]  formal charge (RDKit) = {formal_charge}")

        if mol_raw.GetNumConformers() == 0 or not mol_raw.GetConformer().Is3D():
            raise RuntimeError("No 3D coordinates -- Step 2 output is invalid.")
        print(f"    [3D]    3D coordinates confirmed ({mol_raw.GetNumAtoms()} atoms)")

        lig_dir   = WORKDIR / f"ligand_{lig_num:04d}_{lig_name}"
        lig_dir.mkdir(exist_ok=True)
        lig_sdf   = lig_dir / "ligand.sdf"
        lig_pdbqt = lig_dir / "ligand.pdbqt"

        w = Chem.SDWriter(str(lig_sdf))
        w.SetKekulize(False)
        w.write(mol_raw)
        w.close()

        _ob_sdf_to_pdbqt(lig_sdf, lig_pdbqt)
        lig_charge = _pdbqt_charge_stats(lig_pdbqt)

        print(f"    [chg]   parsed ligand PDBQT atoms = {lig_charge['n_atoms']}")
        print(f"    [chg]   non-zero partial charges  = {lig_charge['nonzero_atoms']}")
        print(f"    [chg]   summed PDBQT charge       = {lig_charge['sum_charge']:.3f}")

        v = Vina(sf_name=scoring_fn, cpu=cpu_cores if cpu_cores > 0 else 0, seed=seed, verbosity=0)
        v.set_receptor(str(prot_pdbqt))
        v.set_ligand_from_file(str(lig_pdbqt))
        v.compute_vina_maps(center=center, box_size=size)

        t0 = time.time()
        v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
        elapsed = time.time() - t0

        energies = v.energies(n_poses=n_poses)
        if len(energies) == 0:
            raise RuntimeError("Vina returned zero poses.")
        best_aff = float(energies[0][0])

        if affinity_cutoff is not None and best_aff > affinity_cutoff:
            print(f"    [skip]  best affinity {best_aff:.3f} > cutoff {affinity_cutoff:.3f}")
            continue

        out_pdbqt = lig_dir / "docked_poses.pdbqt"
        out_sdf   = lig_dir / "docked_poses.sdf"
        v.write_poses(str(out_pdbqt), n_poses=n_poses, overwrite=True)
        _pdbqt_to_sdf(out_pdbqt, out_sdf)

        docked_mols = [m for m in Chem.SDMolSupplier(str(out_sdf), removeHs=False) if m is not None]
        if not docked_mols:
            raise RuntimeError("Docked pose conversion failed; SDF is empty.")

        best_pose = docked_mols[0]
        for k, vprop in mol_raw.GetPropsAsDict().items():
            best_pose.SetProp(str(k), str(vprop))
        best_pose.SetProp("_Name", lig_name)
        best_pose.SetProp("Docking_Affinity_kcal_per_mol", f"{best_aff:.4f}")
        best_pose.SetProp("Docking_Run", run_name)
        best_pose.SetProp("Ligand_Formal_Charge", str(formal_charge))
        best_pose.SetProp("Ligand_PDBQT_Summed_Charge", f"{lig_charge['sum_charge']:.4f}")

        best_pose_sdf = lig_dir / "best_pose.sdf"
        wbest = Chem.SDWriter(str(best_pose_sdf))
        wbest.write(best_pose)
        wbest.close()

        if export_pose_pdbs:
            try:
                obc = ob.OBConversion()
                obc.SetInAndOutFormats("pdbqt", "pdb")
                om = ob.OBMol()
                obc.ReadFile(om, str(out_pdbqt))
                obc.WriteFile(om, str(lig_dir / "docked_poses.pdb"))
            except Exception:
                pass

        best_pose_mols.append(best_pose)
        summary_rows.append({
            "Ligand": lig_name,
            "Best_Affinity_kcal_per_mol": best_aff,
            "Elapsed_s": elapsed,
            "Formal_Charge_RDKit": formal_charge,
            "PDBQT_Summed_Charge": lig_charge["sum_charge"],
            "PDBQT_NonzeroChargeAtoms": lig_charge["nonzero_atoms"],
            "Formula": formula,
            "MW": mw,
            "LogP": logp,
            "HBD": hbd,
            "HBA": hba,
            "RotB": rotb,
            "TPSA": tpsa,
            "Best_Pose_SDF": str(best_pose_sdf),
        })

        print(f"    [dock]  best affinity = {best_aff:.3f} kcal/mol  |  elapsed = {elapsed:.1f}s")

    except Exception as e:
        failed_rows.append((lig_name, str(e)))
        print(f"    [FAIL]  {e}")
        if not skip_failed:
            raise


# ---------------------------------------------------------------------------
#  STEP 7 : Ranking + Export Top 10
# ---------------------------------------------------------------------------

_print_header("STEP 7/7", "Ranking, Export, and Download")

import pandas as pd

summary_csv = WORKDIR / f"{run_name}_summary.csv"
top10_sdf   = WORKDIR / f"{run_name}_top10_ranked_best_poses.sdf"
allbest_sdf = WORKDIR / f"{run_name}_all_best_poses.sdf"
zip_path    = WORKDIR / f"{run_name}_results.zip"

if summary_rows:
    df = pd.DataFrame(summary_rows).sort_values("Best_Affinity_kcal_per_mol", ascending=True).reset_index(drop=True)
    df["Rank"] = np.arange(1, len(df) + 1)
    cols = ["Rank"] + [c for c in df.columns if c != "Rank"]
    df = df[cols]
    df.to_csv(summary_csv, index=False)
    print(f"  Summary CSV -> {summary_csv}")

    # all best poses
    if export_merged_sdf:
        pose_map = {m.GetProp("_Name"): m for m in best_pose_mols if m.HasProp("_Name")}
        w_all = Chem.SDWriter(str(allbest_sdf))
        for lig_name in df["Ligand"].tolist():
            m = pose_map.get(lig_name)
            if m is not None:
                w_all.write(m)
        w_all.close()
        print(f"  All best poses SDF -> {allbest_sdf}")

        # top 10 best poses
        top10_names = df.head(10)["Ligand"].tolist()
        w_top = Chem.SDWriter(str(top10_sdf))
        for lig_name in top10_names:
            m = pose_map.get(lig_name)
            if m is not None:
                w_top.write(m)
        w_top.close()
        print(f"  Top 10 ranked best-pose SDF -> {top10_sdf}")

    if len(df) > 0:
        print("\n  Top ranked ligands")
        print(df.head(10).to_string(index=False))

else:
    print("  WARNING: No successful docking results were produced.")

if failed_rows:
    failed_csv = WORKDIR / f"{run_name}_failed.csv"
    pd.DataFrame(failed_rows, columns=["Ligand", "Reason"]).to_csv(failed_csv, index=False)
    print(f"\n  Failed ligand log -> {failed_csv}")

# zip the whole run directory
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in WORKDIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(WORKDIR))
print(f"  ZIP package -> {zip_path}")

from google.colab import files as colab_files
for fp in [summary_csv, allbest_sdf, top10_sdf, zip_path]:
    if fp and Path(fp).exists():
        print(f"  Downloading {Path(fp).name} ...")
        colab_files.download(str(fp))

print("\n" + "=" * 72)
print("  DOCKING COMPLETE")
print("=" * 72)
print(f"  Receptor PDBQT           : {prot_pdbqt}")
print(f"  Ranked summary CSV       : {summary_csv if summary_rows else 'not generated'}")
print(f"  Top 10 best-pose SDF     : {top10_sdf if summary_rows else 'not generated'}")
print(f"  Whole run ZIP            : {zip_path}")
print(f"  Total elapsed            : {time.time() - t0_total:.1f} s")
print("=" * 72)
